In [3]:
import pandas as pd


url = "https://docs.google.com/spreadsheets/d/1X3_n9FTVpYQ4L0CfvvyZzdyZC5WhN3CO/export?format=xlsx"

import pandas as pd
df = pd.read_excel(url)

# Create empty lists
texts = []
labels = []

# Mapping pairs
pairs = [
    ('teaching', 'teaching.1'),
    ('coursecontent', 'coursecontent.1'),
    ('examination', 'Examination'),
    ('labwork', 'labwork.1'),
    ('library_facilities', ' library_facilities'),
    ('extracurricular', 'extracurricular.1')
]

# Loop through each pair
for label_col, text_col in pairs:
    texts.extend(df[text_col].astype(str))
    labels.extend(df[label_col])

# New clean dataset
new_df = pd.DataFrame({
    'feedback': texts,
    'label': labels
})

print(new_df.head())

                                            feedback  label
0  teacher are punctual but they should also give...    0.0
1                                              Good     1.0
2  Excellent lectures are delivered by teachers a...    1.0
3                                               Good    1.0
4  teachers give us all the information required ...    1.0


In [4]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

new_df['cleaned'] = new_df['feedback'].apply(clean_text)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(new_df['cleaned'])
y = new_df['label']
print(new_df.isnull().sum())
new_df = new_df.dropna(subset=['feedback', 'label'])
new_df['label'] = new_df['label'].astype(int)
new_df = new_df[new_df['feedback'].str.strip() != ""]
X = vectorizer.fit_transform(new_df['cleaned'])
y = new_df['label']

feedback    0
label       5
cleaned     0
dtype: int64


In [6]:
# 🔹 INSTALL + IMPORT VADER
!pip install nltk

import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# 🔹 FIX LABELS (remove neutral & map -1 → 0)
new_df = new_df[new_df['label'] != 0]
new_df['label'] = new_df['label'].replace(-1, 0)

print("Original labels:", new_df['label'].unique())

# 🔥 AUTOMATIC SENTIMENT USING VADER
def vader_label(text):
    score = sia.polarity_scores(str(text))
    return 1 if score['compound'] >= 0 else 0

# Use VADER to improve/validate labels
new_df['auto_label'] = new_df['feedback'].apply(vader_label)

new_df['label'] = new_df['auto_label']

print("Labels after VADER:", new_df['label'].unique())

#  FEATURE EXTRACTION
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english')
X = vectorizer.fit_transform(new_df['cleaned'])
y = new_df['label']

# 🔹 TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 🔹 MODEL
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=300)
model.fit(X_train, y_train)

# 🔹 EVALUATION
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Predicted labels:", set(y_pred))

# 🔥 DEMO COMPARISON (VERY IMPRESSIVE)
sample = "daam this calss is fun"

ml_pred = model.predict(vectorizer.transform([sample]))[0]
vader_pred = vader_label(sample)

print("\nSample:", sample)
print("ML Prediction:", "Positive" if ml_pred == 1 else "Negative")
print("VADER Prediction:", "Positive" if vader_pred == 1 else "Negative")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Original labels: [1 0]
Labels after VADER: [1 0]
Accuracy: 0.9528795811518325
Predicted labels: {np.int64(1)}

Sample: daam this calss is fun
ML Prediction: Positive
VADER Prediction: Positive


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

In [ ]:
from wordcloud import WordCloud

text = " ".join(new_df['cleaned'])

wc = WordCloud(width=800, height=400).generate(text)

plt.imshow(wc)
plt.axis("off")
plt.show()

In [ ]:
sample = ["the teacher is teaching very bad"]
sample_clean = [clean_text(sample[0])]

prediction = model.predict(vectorizer.transform(sample_clean))

print("Prediction:", "Positive" if prediction[0] == 1 else "Negative")